In [2]:
# json2txt
import json
import cv2
import os

# 資料來源路徑
data_dir = './Bug_Detection/'
# 類別名稱所對應的索引
label2idx = {'bug':0}
# 各類別對應畫記的矩形框顏色以作區別 (BGR)
idx2color = { 0:(255,0,0)}
# 待處理的資料集清單
phase_list = ['train', 'val']
for phase_name in phase_list:
    print(phase_name)
    phase_dir = data_dir + phase_name + '/'
    json_dir = phase_dir + 'annotations/'  # 存放 labelme 標註檔案(.json)的資料夾
    img_dir = phase_dir + 'images/'  # 存放影像的資料夾，名稱須為 images
    # 新增YOLO所需的標註檔案(.txt)資料夾，名稱須為 labels
    label_dir = phase_dir + 'labels/'
    if(not os.path.exists(label_dir)):
        os.mkdir(label_dir)
    # 新增將標註框/輪廓畫在原圖上的偵測 ground truth 資料夾做為比對參考
    GT_dir = phase_dir + 'img_gt/'
    if(not os.path.exists(GT_dir)):
        os.mkdir(GT_dir)
    # 對於 images 資料夾中的每張影像檔案
    img_list = os.listdir(img_dir)
    for img_name in img_list:
        # 讀取影像資訊
        img = cv2.imread(img_dir + img_name)
        height, width, depth = img.shape  # rows, cols, channels
        # 去除影像的副檔名
        case_name = '.'.join(img_name.split('.')[:-1])
        # 準備寫出符合 YOLO 規範的標註資訊
        txt_name = case_name + '.txt'
        f_txt = open(label_dir + txt_name, 'w')
        # 讀取對應的 .json 標註檔案
        json_name = case_name + '.json'
        with open(json_dir+json_name) as f_json:
            info = json.load(f_json)
        # 讀取標註資訊
        bbox_info = info['shapes']
        for bbox in bbox_info:
            label_name = bbox['label']
            label_idx = label2idx[label_name]
            set_color = idx2color[label_idx]
            pts = bbox['points']   # pts = [[左上角 x, 左上角 y], [右下角 x, 右下角 y]]
            pt_1 = (int(pts[0][0]), int(pts[0][1]))  # 偵測框的左上點座標
            pt_2 = (int(pts[1][0]), int(pts[1][1]))  # 偵測框的右下點座標
            # 在對應的影像上畫上偵測矩形框
            cv2.rectangle(img, pt_1, pt_2, set_color, 3)
            # 計算偵測矩形框的相對中心點座標
            xc = ((pt_1[0]+pt_2[0])/2)/width
            yc = ((pt_1[1]+pt_2[1])/2)/height
            # 計算偵測矩形框的相對寬與高
            w = abs(pt_2[0]-pt_1[0])/width
            h = abs(pt_2[1]-pt_1[1])/height
            # 資料以空格區隔，不同目標物件以換行區隔
            instance_info = f'{label_idx} {xc} {yc} {w} {h}\n'
            f_txt.write(instance_info)
        f_txt.close()
        cv2.imwrite(GT_dir+img_name, img)

train
val


In [ ]:
from ultralytics import YOLO

if __name__ == '__main__':
    # 1. 確保釋放之前的資源
    import torch
    torch.cuda.empty_cache()

    model = YOLO('yolo11s.pt')

    # 2. 4070 專用設定
    results = model.train(
        data='data_Bug.yaml', 
        epochs=100, 
        imgsz=1024,      # <--- 4070 的黃金解析度 (比 640 清楚非常多！)
        batch=8,         # <--- 4070 12G 跑 1024 + batch 8 應該游刃有餘
        workers=0,       # <--- 【絕對關鍵】設為 0，解決 Kernel Crash 的元兇
        name='bug_4070_power',
        amp=True,        # 開啟混合精度加速 (省顯存)
        cache=False      # 不要把圖片快取到 RAM，避免系統記憶體爆炸
    )

In [8]:
from ultralytics import YOLO

if __name__ == '__main__':
    # 1. 載入更強的 Medium 模型 (會自動下載)
    model = YOLO('yolo11m.pt') 

    # 2. 開始訓練 (衝分模式)
    results = model.train(
        data='data_Bug.yaml', 
        epochs=100, 
        imgsz=1024,      # 維持高解析度 (1024)
        batch=4,         # M版模型比較大，Batch 設 4 比較安全 (4070 12G)
        workers=0,       # 鎖住 Workers 避免記憶體崩潰
        name='bug_medium_1024_aug', # 改個名字紀錄這是「增強版」
        amp=True,
        cache=False,     # <--- 這裡原本少了一個逗號！
        
        # --- 暴力增強參數區 ---
        flipud=0.5,      # 上下翻轉 (介殼蟲上下看都一樣，開這個很有用)
        degrees=180,     # 旋轉 180 度 (模擬各種角度的蟲)
        mixup=0.1        # 混合圖片 (讓蟲跟背景融合，增加難度以提升抗干擾力)
    )

New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.233  Python-3.13.9 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_Bug.yaml, degrees=180, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=bug_medium_1024_aug, nbs=64,

In [13]:
# YOLO Detection
from ultralytics import YOLO
import cv2

if(__name__=='__main__'):
    # 載入模型
    model = YOLO('runs/detect/bug_medium_1024_aug/weights/best.pt')
    # 讀取鏡頭
    cap = cv2.VideoCapture(1)  # 0通常為內建前鏡頭，1以上通常為後鏡頭或外接鏡頭。
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))    
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'Webcam : {width} x {height}')

    while(cap.isOpened()):
        ret, frame = cap.read()
        if(not ret):
            print('No camera found!')
            break
        #frame = cv2.flip(frame, 1)  # 鏡像反轉: 前鏡頭需要鏡像反轉，後鏡頭則註解此行。
        # 模型推論 inference
        results = model(frame, conf=0.2, iou=0.25)
        # 模型推論 inference
        cv2.imshow('YOLO Detection', results[0].plot())  
        k = cv2.waitKey(1) & 0xFF
        if(k==27):  # 按ESC鍵結束程式
            print('ESC 結束')
            break
    
    cap.release()
    cv2.destroyAllWindows()

Webcam : 640 x 480

0: 768x1024 (no detections), 83.2ms
Speed: 5.6ms preprocess, 83.2ms inference, 1.1ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 66.6ms
Speed: 4.9ms preprocess, 66.6ms inference, 1.0ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 59.1ms
Speed: 4.0ms preprocess, 59.1ms inference, 0.7ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 59.4ms
Speed: 6.9ms preprocess, 59.4ms inference, 1.0ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 56.8ms
Speed: 3.2ms preprocess, 56.8ms inference, 0.7ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 54.9ms
Speed: 4.0ms preprocess, 54.9ms inference, 0.6ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 52.6ms
Speed: 3.2ms preprocess, 52.6ms inference, 1.3ms postprocess per image at shape (1, 3, 768, 1024)

0: 768x1024 (no detections), 49.9ms

In [1]:
from ultralytics import YOLO

if __name__ == '__main__':
    # 載入你訓練好的模型
    model = YOLO('./runs/detect/bug_medium_1024_aug/weights/best.pt')

    # 執行預測
    results = model.predict(
        source='./Bug_Detection/val/images/',
        conf=0.25,       # 稍微提高一點信心度，過濾雜訊
        iou=0.45,        # <--- 改回 0.45，避免密集的蟲被誤刪
        save=True,       # 存圖
    )
    print("預測完成！請去 runs/detect/predict 資料夾看結果")


image 1/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_141903_224.JPG: 768x1024 1 Bug, 39.3ms
image 2/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_141916_225.JPG: 768x1024 (no detections), 16.3ms
image 3/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_141946_226.JPG: 768x1024 3 Bugs, 16.4ms
image 4/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_143118_230.JPG: 768x1024 6 Bugs, 16.3ms
image 5/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_143141_231.JPG: 768x1024 7 Bugs, 17.0ms
image 6/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_143305_232.JPG: 768x1024 12 Bugs, 16.9ms
image 7/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_143333_233.JPG: 768x1024 25 Bugs, 17.0ms
image 8/75 c:\Users\yweiw\OneDrive\Desktop\yolo\Bug_Detection\val\images\2024_0329_143400_234.JPG: 768x1024 31 Bugs, 17.0ms
imag